# 07 — Pascal-Part-116 benchmark and literature comparison

## Objective

Audit the completed experiments, preserve validation-only model selection, and
separate two fundamentally different reports:

1. the **internal query-level ablation**, using the metric saved by notebooks 01–05;
2. a **benchmark-style semantic evaluation**, following the official OV-PARTS
   confusion-matrix and 74-seen/42-unseen class split as closely as this
   parent-mask-conditioned model permits.

Published values are reference values, not leaderboard-equivalent claims. The
notebook never substitutes mean per-query IoU for semantic-class mIoU.


## 1. Setup and output contract

Run this notebook only after notebooks 01–06 have completed. It does not train,
select, or modify a model. All generated files are written beneath
`training_results_corrected/benchmark_comparison/`.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "datasets").is_dir() and (path / "final_model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULT_ROOT = PROJECT_ROOT / "training_results_corrected"
OUTPUT_DIR = RESULT_ROOT / "benchmark_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = (
    "baseline_object_mask",
    "fixed_uvd",
    "query_gated_uvd",
    "rotation_consistent",
    "geometry_dropout",
)

print("Project:", PROJECT_ROOT)
print("Benchmark outputs:", OUTPUT_DIR.relative_to(PROJECT_ROOT))


Project: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation
Benchmark outputs: training_results_corrected/benchmark_comparison


## 2. Completion, selection, and protocol audit

The selected model must come from notebook 06 and therefore from maximum
`validation_seen` IoU. Test results are not used for checkpoint selection.


In [2]:
import pandas as pd

required = ("completion.json", "summary.csv", "detailed_predictions.csv", "ui_model.pt")
missing = [
    str((RESULT_ROOT / experiment / name).relative_to(PROJECT_ROOT))
    for experiment in EXPERIMENTS for name in required
    if not (RESULT_ROOT / experiment / name).is_file()
]
registry_path = RESULT_ROOT / "model_registry.json"
if not registry_path.is_file():
    missing.append(str(registry_path.relative_to(PROJECT_ROOT)))
if missing:
    raise RuntimeError("Run notebooks 01–06 completely first. Missing:\n- " + "\n- ".join(missing))

registry = json.loads(registry_path.read_text())
assert registry["selection_rule"] == "maximum validation_seen IoU; no test metric used for selection"
selected_model = registry["selected_model"]
selected_checkpoint = PROJECT_ROOT / registry["selected_checkpoint"]
if not selected_checkpoint.is_file():
    raise FileNotFoundError(selected_checkpoint)

checkpoint_sha256 = hashlib.sha256(selected_checkpoint.read_bytes()).hexdigest()
completion = {
    experiment: json.loads((RESULT_ROOT / experiment / "completion.json").read_text())
    for experiment in EXPERIMENTS
}
completion_table = pd.DataFrame(completion).T
display(completion_table)
print("Validation-selected model:", selected_model)
print("Checkpoint:", selected_checkpoint.relative_to(PROJECT_ROOT))
print("SHA-256:", checkpoint_sha256)


,experiment,status,stop_reason,epochs_completed,selected_epoch,best_validation_iou,checkpoint
baseline_object_mask,baseline_object_mask,trained,early_stopping_validation_iou_plateau,20,20,0.288043,training_results_corrected/baseline_object_mas...
fixed_uvd,fixed_uvd,trained,early_stopping_validation_iou_plateau,20,15,0.291171,training_results_corrected/fixed_uvd/best.pt
query_gated_uvd,query_gated_uvd,trained,early_stopping_validation_iou_plateau,20,15,0.290894,training_results_corrected/query_gated_uvd/bes...
rotation_consistent,rotation_consistent,trained,early_stopping_validation_iou_plateau,16,14,0.293719,training_results_corrected/rotation_consistent...
geometry_dropout,geometry_dropout,trained,early_stopping_validation_iou_plateau,22,17,0.287762,training_results_corrected/geometry_dropout/be...


Validation-selected model: rotation_consistent
Checkpoint: models/final_study/best_model.pt
SHA-256: fe361285eaa65b5844b705c3d8715f781607e7ceba549dff0cf63c3e65bd5ebd


In [3]:
from datasets.metadata import PART_CATEGORIES, PARTS_BY_OBJECT, UNSEEN_OBJECT_NAMES

seen_ids = [p["part_id"] for p in PART_CATEGORIES if p["evaluation_split"] == "seen"]
unseen_ids = [p["part_id"] for p in PART_CATEGORIES if p["evaluation_split"] == "unseen"]
assert len(PART_CATEGORIES) == 116
assert (len(seen_ids), len(unseen_ids)) == (74, 42)

protocol_audit = {
    "existing_metric": "arithmetic mean of thresholded binary IoU over present part-query samples",
    "existing_aggregation": "per query/sample; not per class and not global pixels",
    "existing_is_semantic_miou": False,
    "existing_threshold": 0.5,
    "seen_definition": "part classes belonging to 11 trained parent-object categories",
    "unseen_definition": "42 part classes belonging to bird, car, dog, motorbike, and sheep",
    "official_class_partition_counts": {"seen": len(seen_ids), "unseen": len(unseen_ids)},
    "model_input": "ground-truth semantic parent-object mask",
    "closest_protocol": "parent-mask-conditioned / Oracle-Obj-like",
    "strict_oracle_obj_equivalence": False,
    "why_not_strict": (
        "The model is trained as independent binary present-part queries at 224 px; "
        "the official evaluator predicts a multiclass part map per oracle object region."
    ),
    "benchmark_decoding": "argmax across all applicable part-query probabilities inside each oracle parent mask",
    "benchmark_threshold_tuning": "none",
    "selection_rule": registry["selection_rule"],
    "selected_model": selected_model,
    "selected_checkpoint": str(selected_checkpoint.relative_to(PROJECT_ROOT)),
    "selected_checkpoint_sha256": checkpoint_sha256,
}
(OUTPUT_DIR / "protocol_audit.json").write_text(json.dumps(protocol_audit, indent=2) + "\n")
display(pd.Series(protocol_audit, name="value").to_frame())


,value
existing_metric,arithmetic mean of thresholded binary IoU over...
existing_aggregation,per query/sample; not per class and not global...
existing_is_semantic_miou,False
existing_threshold,0.5
seen_definition,part classes belonging to 11 trained parent-ob...
unseen_definition,"42 part classes belonging to bird, car, dog, m..."
official_class_partition_counts,"{'seen': 74, 'unseen': 42}"
model_input,ground-truth semantic parent-object mask
closest_protocol,parent-mask-conditioned / Oracle-Obj-like
strict_oracle_obj_equivalence,False


## 3. Ground-truth parent-mask repair audit

The training dataset currently contains a defensive branch that unions the
resized target part into the resized parent mask when their overlap disappears.
This cell deterministically counts activations across every training/evaluation
query. Zero activations means that branch could not have affected the completed
runs; a nonzero count requires an explicit limitation and prevents a strict
benchmark claim.


In [4]:
from collections import defaultdict
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

from final_model.training_core import preprocess_mask

records_by_image = defaultdict(list)
for split in ("train_seen", "validation_seen", "test_seen", "test_unseen"):
    records = json.loads((PROJECT_ROOT / "data" / "processed" / f"{split}.json").read_text())
    for record in records:
        key = (record["object_annotation_path"], record["part_annotation_path"])
        records_by_image[key].append((split, record))

repair_rows = []
audited_queries = 0
for (object_path, part_path), items in tqdm(records_by_image.items(), desc="repair audit"):
    with Image.open(PROJECT_ROOT / object_path) as handle:
        object_labels = np.asarray(handle).copy()
    with Image.open(PROJECT_ROOT / part_path) as handle:
        part_labels = np.asarray(handle).copy()
    height, width = object_labels.shape
    for split, record in items:
        parent = torch.from_numpy(object_labels == int(record["object_id"]))
        target = torch.from_numpy(part_labels == int(record["part_id"])) & parent
        resized_parent = preprocess_mask(parent, height, width, 224)
        resized_target = preprocess_mask(target, height, width, 224)
        activated = bool(resized_target.any() and not (resized_target & resized_parent).any())
        audited_queries += 1
        if activated:
            repair_rows.append({
                "split": split,
                "sample_id": record["sample_id"],
                "image_id": record["image_id"],
                "object_name": record["object_name"],
                "part_name": record["part_name"],
            })

repair_cases = pd.DataFrame(
    repair_rows,
    columns=("split", "sample_id", "image_id", "object_name", "part_name"),
)
repair_cases.to_csv(OUTPUT_DIR / "parent_mask_repair_cases.csv", index=False)
repair_audit = {
    "audited_queries": audited_queries,
    "repair_activations": len(repair_cases),
    "strict_training_input_hygiene": len(repair_cases) == 0,
}
(OUTPUT_DIR / "parent_mask_repair_audit.json").write_text(json.dumps(repair_audit, indent=2) + "\n")
display(pd.Series(repair_audit, name="value").to_frame())
if len(repair_cases):
    display(repair_cases.head(20))
    print("CAUTION: report the repair as a limitation; do not claim strict benchmark equivalence.")
else:
    print("The repair branch never activated, so it did not affect these runs.")


repair audit:   0%|          | 0/6633 [00:00<?, ?it/s]

,value
audited_queries,41363
repair_activations,76
strict_training_input_hygiene,False


,split,sample_id,image_id,object_name,part_name
0,train_seen,train:2008_000041:part_93,2008_000041,person,hair
1,train_seen,train:2008_000051:part_21,2008_000051,bottle,cap
2,train_seen,train:2008_001080:part_89,2008_001080,person,nose
3,train_seen,train:2008_001080:part_92,2008_001080,person,mouth
4,train_seen,train:2008_001306:part_93,2008_001306,person,hair
5,train_seen,train:2008_001577:part_84,2008_001577,person,eye
6,train_seen,train:2008_001659:part_90,2008_001659,person,ear
7,train_seen,train:2008_001690:part_92,2008_001690,person,mouth
8,train_seen,train:2008_002067:part_45,2008_002067,cat,torso
9,train_seen,train:2008_003334:part_90,2008_003334,person,ear


CAUTION: report the repair as a limitation; do not claim strict benchmark equivalence.


## 4. Strictly internal ablation

These values retain the notebooks' original metric: mean thresholded binary IoU
over present part-query samples. The harmonic mean below is useful internally,
but it is **not** the official semantic-class h-IoU.


In [5]:
internal_rows = []
for experiment in EXPERIMENTS:
    summary = pd.read_csv(RESULT_ROOT / experiment / "summary.csv").set_index("split")
    seen = float(summary.loc["test_seen", "iou"])
    unseen = float(summary.loc["test_unseen", "iou"])
    internal_rows.append({
        "method": experiment,
        "selected_epoch": int(summary["selected_epoch"].iloc[0]),
        "validation_query_mean_iou": float(summary["validation_iou"].iloc[0]),
        "test_seen_query_mean_iou": seen,
        "test_unseen_query_mean_iou": unseen,
        "internal_h_iou": 2 * seen * unseen / (seen + unseen) if seen + unseen else float("nan"),
        "metric": "mean per-present-query binary IoU @ 0.5",
    })
internal_ablation = pd.DataFrame(internal_rows).sort_values(
    "validation_query_mean_iou", ascending=False
)
internal_ablation.to_csv(OUTPUT_DIR / "internal_ablation.csv", index=False)
display(internal_ablation.round(4))


,method,selected_epoch,validation_query_mean_iou,test_seen_query_mean_iou,test_unseen_query_mean_iou,internal_h_iou,metric
3,rotation_consistent,14,0.2937,0.3028,0.2712,0.2861,mean per-present-query binary IoU @ 0.5
1,fixed_uvd,15,0.2912,0.3015,0.2493,0.2729,mean per-present-query binary IoU @ 0.5
2,query_gated_uvd,15,0.2909,0.3028,0.2456,0.2712,mean per-present-query binary IoU @ 0.5
0,baseline_object_mask,20,0.2880,0.2956,0.2243,0.2551,mean per-present-query binary IoU @ 0.5
4,geometry_dropout,17,0.2878,0.2972,0.2415,0.2665,mean per-present-query binary IoU @ 0.5


## 5. Official-style Oracle-Obj evaluation adapter

The official OV-PARTS evaluator constructs a multiclass map with `argmax`, masks
it to the oracle parent-object region, accumulates TP/FP/FN by semantic part
class, and reports mean class IoU for 74 seen and 42 unseen classes. The adapter
below follows those operations at original image resolution.

Protocol difference: this model was optimized using independent binary queries
only for parts present in each training object. Consequently the result is
reported as **parent-mask-conditioned / Oracle-Obj-like**, not as an official
Oracle-Obj leaderboard entry.


In [6]:
from contextlib import nullcontext
import torch.nn.functional as F

from final_model.inference import load_predictor
from final_model.training_core import preprocess_image, relative_uvd, resize_info

def image_visual_features(predictor, image_chw):
    _, model_image = preprocess_image(image_chw.cpu(), predictor.config.image_size)
    model_image = model_image.unsqueeze(0).to(predictor.device)
    context = torch.autocast("cuda", dtype=torch.float16) if predictor.device.type == "cuda" else nullcontext()
    with torch.inference_mode(), context:
        visual = predictor.model.visual_projection(predictor.model.visual_features(model_image))
    return visual

def applicable_part_probabilities(predictor, visual, parent_hw, queries):
    height, width = parent_hw.shape
    size = predictor.config.image_size
    model_mask = preprocess_mask(torch.from_numpy(parent_hw), height, width, size).float()
    u, v, d = relative_uvd(model_mask.bool())
    text_embeddings = predictor.text(list(queries))
    count = len(queries)
    context = torch.autocast("cuda", dtype=torch.float16) if predictor.device.type == "cuda" else nullcontext()
    with torch.inference_mode(), context:
        text_map = predictor.model.text_projection(text_embeddings)[:, :, None, None]
        text_map = text_map.expand(-1, -1, *visual.shape[-2:])
        mask_low = F.interpolate(
            model_mask[None].to(predictor.device), visual.shape[-2:], mode="nearest"
        ).expand(count, -1, -1, -1)
        geometry = torch.cat([
            F.interpolate(x[None].to(predictor.device), visual.shape[-2:], mode="bilinear", align_corners=False)
            for x in (u, v, d)
        ], dim=1).expand(count, -1, -1, -1) * mask_low
        if predictor.model.experiment == "baseline_object_mask":
            gates = torch.zeros(
                count, 3, device=text_embeddings.device, dtype=text_embeddings.dtype
            )
        elif predictor.model.experiment == "fixed_uvd":
            gates = torch.ones(
                count, 3, device=text_embeddings.device, dtype=text_embeddings.dtype
            )
        else:
            assert predictor.model.geometry_gate is not None
            gates = torch.sigmoid(predictor.model.geometry_gate(text_embeddings))
        geometry = geometry * gates[:, :, None, None]
        low_logits = predictor.model.decoder(torch.cat([
            visual.expand(count, -1, -1, -1), text_map, mask_low, geometry
        ], dim=1))
        probability = torch.sigmoid(F.interpolate(
            low_logits, (size, size), mode="bilinear", align_corners=False
        )).float()
        info = resize_info(height, width, size)
        top, left = int(info["top"]), int(info["left"])
        new_h, new_w = int(info["new_h"]), int(info["new_w"])
        probability = probability[:, :, top:top + new_h, left:left + new_w]
        probability = F.interpolate(probability, (height, width), mode="bilinear", align_corners=False)
    return probability[:, 0].cpu().numpy()


In [7]:
import time

if not torch.cuda.is_available():
    raise RuntimeError("Benchmark inference requires CUDA, like the training notebooks")

torch.cuda.reset_peak_memory_stats()
benchmark_started = time.perf_counter()
predictor = load_predictor(selected_checkpoint, device="cuda:0")
raw_root = PROJECT_ROOT / "data" / "raw" / "PascalPart116"
image_dir = raw_root / "images" / "val"
object_dir = raw_root / "annotations_detectron2_obj" / "val"
part_dir = raw_root / "annotations_detectron2_part" / "val"

part_ids_by_object = {
    object_id: [p["part_id"] for p in PART_CATEGORIES if p["object_id"] == object_id]
    for object_id in PARTS_BY_OBJECT
}
confusion = np.zeros((116, 116), dtype=np.int64)
evaluated_objects = 0

image_paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"} and not p.name.startswith("."))
for image_path in tqdm(image_paths, desc="official-style val inference"):
    stem = image_path.stem
    with Image.open(image_path) as handle:
        image_np = np.asarray(handle.convert("RGB")).copy()
    with Image.open(object_dir / f"{stem}.png") as handle:
        object_labels = np.asarray(handle).copy()
    with Image.open(part_dir / f"{stem}.png") as handle:
        part_labels = np.asarray(handle).copy()
    image_chw = torch.from_numpy(image_np.transpose(2, 0, 1)).to(torch.uint8)
    visual = image_visual_features(predictor, image_chw)

    for object_id in sorted(set(np.unique(object_labels)) & set(part_ids_by_object)):
        class_ids = part_ids_by_object[int(object_id)]
        parent = object_labels == int(object_id)
        valid = parent & np.isin(part_labels, class_ids)
        if not valid.any():
            continue
        queries = [PART_CATEGORIES[class_id]["part_name"] for class_id in class_ids]
        scores = applicable_part_probabilities(predictor, visual, parent, queries)
        predicted = np.asarray(class_ids, dtype=np.int64)[scores.argmax(axis=0)]
        gt = part_labels[valid].astype(np.int64)
        pred = predicted[valid]
        confusion += np.bincount(116 * gt + pred, minlength=116 * 116).reshape(116, 116)
        evaluated_objects += 1

benchmark_seconds = time.perf_counter() - benchmark_started
benchmark_peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3
np.save(OUTPUT_DIR / "semantic_confusion_matrix.npy", confusion)
tp = np.diag(confusion).astype(np.int64)
gt_pixels = confusion.sum(axis=1)
pred_pixels = confusion.sum(axis=0)
fp = pred_pixels - tp
fn = gt_pixels - tp
union = tp + fp + fn
iou = np.divide(tp, union, out=np.full(116, np.nan), where=union > 0)

per_class_iou = pd.DataFrame([{**part, "tp": int(tp[part["part_id"]]),
    "fp": int(fp[part["part_id"]]), "fn": int(fn[part["part_id"]]),
    "gt_pixels": int(gt_pixels[part["part_id"]]), "pred_pixels": int(pred_pixels[part["part_id"]]),
    "iou": float(iou[part["part_id"]])} for part in PART_CATEGORIES])
per_class_iou.to_csv(OUTPUT_DIR / "per_class_iou.csv", index=False)

valid_seen = per_class_iou.query("evaluation_split == 'seen' and gt_pixels > 0")
valid_unseen = per_class_iou.query("evaluation_split == 'unseen' and gt_pixels > 0")
seen_miou = float(valid_seen["iou"].mean())
unseen_miou = float(valid_unseen["iou"].mean())
h_iou = 2 * seen_miou * unseen_miou / (seen_miou + unseen_miou)
benchmark_summary = {
    "method": selected_model,
    "protocol": "parent-mask-conditioned / Oracle-Obj-like",
    "strict_leaderboard_equivalent": False,
    "seen_miou": seen_miou,
    "unseen_miou": unseen_miou,
    "h_iou": h_iou,
    "valid_seen_classes": len(valid_seen),
    "valid_unseen_classes": len(valid_unseen),
    "evaluated_oracle_object_regions": evaluated_objects,
    "benchmark_seconds_including_model_load": benchmark_seconds,
    "images_per_second_including_model_load": len(image_paths) / benchmark_seconds,
    "peak_gpu_memory_gb": benchmark_peak_gpu_gb,
    "decoder": "argmax across applicable part-query probabilities; no test threshold tuning",
    "checkpoint": str(selected_checkpoint.relative_to(PROJECT_ROOT)),
    "checkpoint_sha256": checkpoint_sha256,
    "parent_mask_repair_activations_during_pipeline_audit": len(repair_cases),
}
(OUTPUT_DIR / "benchmark_summary.json").write_text(json.dumps(benchmark_summary, indent=2) + "\n")
display(pd.Series(benchmark_summary, name="value").to_frame())
display(per_class_iou.sort_values("iou").head(15).round(4))


Using cache found in /home/utn/ojug99ek/.cache/torch/hub/facebookresearch_dinov2_main
/home/utn/ojug99ek/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/utn/ojug99ek/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/utn/ojug99ek/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


official-style val inference:   0%|          | 0/850 [00:00<?, ?it/s]

,value
method,rotation_consistent
protocol,parent-mask-conditioned / Oracle-Obj-like
strict_leaderboard_equivalent,False
seen_miou,0.477781
unseen_miou,0.313102
h_iou,0.378297
valid_seen_classes,73
valid_unseen_classes,41
evaluated_oracle_object_regions,884
benchmark_seconds_including_model_load,27.633792


,part_id,object_id,object_name,part_name,full_name,evaluation_split,tp,fp,fn,gt_pixels,pred_pixels,iou
3,3,0,aeroplane,tail,aeroplane's tail,seen,0,0,5,5,0,0.0000
10,10,1,bicycle,headlight,bicycle's headlight,seen,0,2259,0,0,2259,0.0000
27,27,5,bus,roof,bus's roof,seen,0,81447,1908,1908,81447,0.0000
29,29,5,bus,license plate,bus's license plate,seen,0,822,24,24,822,0.0000
39,39,6,car,license plate,car's license plate,unseen,0,601,124,124,601,0.0000
80,80,13,motorbike,saddle,motorbike's saddle,unseen,0,11026,0,0,11026,0.0000
108,108,18,train,headlight,train's headlight,seen,0,26,648,648,26,0.0000
38,38,6,car,mirror,car's mirror,unseen,11,2837,9342,9353,2848,0.0009
113,113,18,train,roof,train's roof,seen,254,39855,19953,20207,40109,0.0042
28,28,5,bus,mirror,bus's mirror,seen,59,0,13523,13582,59,0.0043


## 6. Verified literature references

Numbers below are percentages transcribed from primary papers. Rows with
different protocols remain separate. PartCATSeg Table 1 supplies the consistent
Oracle-Obj reference block; PCA-Seg Table 2 and HOPS Table 3 are recorded from
their own papers. Because training/evaluation implementations differ, the Ours
row remains explicitly approximate.


In [8]:
literature_rows = [
    # PartCATSeg, CVPR 2025, Table 1 — Oracle-Obj
    ("ZSSeg+", "Oracle-Obj", 54.43, 19.04, 28.21, "PartCATSeg", 2025, "Table 1"),
    ("VLPart", "Oracle-Obj", 42.61, 18.70, 25.99, "PartCATSeg", 2025, "Table 1"),
    ("CLIPSeg", "Oracle-Obj", 48.91, 27.54, 35.24, "PartCATSeg", 2025, "Table 1"),
    ("CAT-Seg", "Oracle-Obj", 43.81, 27.66, 33.91, "PartCATSeg", 2025, "Table 1"),
    ("PartCLIPSeg", "Oracle-Obj", 50.02, 31.67, 38.79, "PartCATSeg", 2025, "Table 1"),
    ("PartCATSeg", "Oracle-Obj", 57.49, 44.88, 50.41, "PartCATSeg", 2025, "Table 1"),
    # PCA-Seg, arXiv 2026, Table 2 — Oracle-Obj
    ("PCA-Seg", "Oracle-Obj", 60.50, 47.10, 52.90, "PCA-Seg", 2026, "Table 2"),
    # HOPS, CVPR 2026, Table 3 — its hierarchical predicted-object pipeline
    ("HOPS", "Pred-All / hierarchical", 54.77, 44.81, 49.29, "HOPS", 2026, "Table 3"),
]
source_urls = {
    "PartCATSeg": "https://openaccess.thecvf.com/content/CVPR2025/papers/Choi_Fine-Grained_Image-Text_Correspondence_with_Cost_Aggregation_for_Open-Vocabulary_Part_Segmentation_CVPR_2025_paper.pdf",
    "PCA-Seg": "https://arxiv.org/abs/2603.17520",
    "HOPS": "https://openaccess.thecvf.com/content/CVPR2026/html/Li_HOPS_Hierarchical_Open-vocabulary_Part_Segmentation_with_Attention-Aware_Filtering_and_Affinity-Guided_CVPR_2026_paper.html",
    "OV-PARTS protocol": "https://github.com/OpenRobotLab/OV_PARTS",
}
literature = pd.DataFrame(literature_rows, columns=(
    "method", "protocol", "seen_miou_percent", "unseen_miou_percent", "h_iou_percent",
    "source", "year", "table",
))
literature["source_url"] = literature["source"].map(source_urls)
ours = pd.DataFrame([{
    "method": "Ours — " + selected_model,
    "protocol": "parent-mask-conditioned / Oracle-Obj-like",
    "seen_miou_percent": 100 * seen_miou,
    "unseen_miou_percent": 100 * unseen_miou,
    "h_iou_percent": 100 * h_iou,
    "source": "this notebook",
    "year": None,
    "table": "benchmark_summary.json",
    "source_url": None,
}])
benchmark_table = pd.concat([literature, ours], ignore_index=True)
benchmark_table.to_csv(OUTPUT_DIR / "literature_comparison.csv", index=False)
pd.DataFrame([{"source": k, "url": v} for k, v in source_urls.items()]).to_csv(
    OUTPUT_DIR / "literature_sources.csv", index=False
)
display(benchmark_table.round(2))

oracle_references = literature[literature["protocol"] == "Oracle-Obj"]
best_reference = oracle_references.sort_values("h_iou_percent", ascending=False).iloc[0]
internal_winner = internal_ablation.iloc[0]
scientific_interpretation = {
    "is_strict_literature_comparison": False,
    "appropriate_claim": "approximate parent-mask-conditioned / Oracle-Obj-like comparison",
    "ours_h_iou_percent": 100 * h_iou,
    "best_oracle_obj_reference_method": best_reference["method"],
    "best_oracle_obj_reference_h_iou_percent": float(best_reference["h_iou_percent"]),
    "difference_from_best_reference_points": 100 * h_iou - float(best_reference["h_iou_percent"]),
    "internally_selected_model": selected_model,
    "internal_validation_winner": internal_winner["method"],
    "repair_audit_passed": len(repair_cases) == 0,
    "conclusion_rule": (
        "Describe competitiveness only against reference values under the stated approximate protocol; "
        "never claim a strict Pred-All or Oracle-Obj leaderboard result."
    ),
}
(OUTPUT_DIR / "scientific_interpretation.json").write_text(
    json.dumps(scientific_interpretation, indent=2) + "\n"
)
display(pd.Series(scientific_interpretation, name="value").to_frame())


,method,protocol,seen_miou_percent,unseen_miou_percent,h_iou_percent,source,year,table,source_url
0,ZSSeg+,Oracle-Obj,54.43,19.04,28.21,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
1,VLPart,Oracle-Obj,42.61,18.70,25.99,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
2,CLIPSeg,Oracle-Obj,48.91,27.54,35.24,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
3,CAT-Seg,Oracle-Obj,43.81,27.66,33.91,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
4,PartCLIPSeg,Oracle-Obj,50.02,31.67,38.79,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
5,PartCATSeg,Oracle-Obj,57.49,44.88,50.41,PartCATSeg,2025,Table 1,https://openaccess.thecvf.com/content/CVPR2025...
6,PCA-Seg,Oracle-Obj,60.50,47.10,52.90,PCA-Seg,2026,Table 2,https://arxiv.org/abs/2603.17520
7,HOPS,Pred-All / hierarchical,54.77,44.81,49.29,HOPS,2026,Table 3,https://openaccess.thecvf.com/content/CVPR2026...
8,Ours — rotation_consistent,parent-mask-conditioned / Oracle-Obj-like,47.78,31.31,37.83,this notebook,None,benchmark_summary.json,None


,value
is_strict_literature_comparison,False
appropriate_claim,approximate parent-mask-conditioned / Oracle-O...
ours_h_iou_percent,37.829697
best_oracle_obj_reference_method,PCA-Seg
best_oracle_obj_reference_h_iou_percent,52.9
difference_from_best_reference_points,-15.070303
internally_selected_model,rotation_consistent
internal_validation_winner,rotation_consistent
repair_audit_passed,False
conclusion_rule,Describe competitiveness only against referenc...


In [9]:
import matplotlib.pyplot as plt

plot_table = benchmark_table.set_index("method")
figure, axes = plt.subplots(1, 2, figsize=(16, 5.5))
plot_table[["seen_miou_percent", "unseen_miou_percent"]].plot.bar(
    ax=axes[0], color=("#38bdf8", "#22c55e")
)
axes[0].set(title="Published references and Ours", ylabel="mIoU (%)", xlabel="")
axes[0].tick_params(axis="x", rotation=35)
internal_ablation.set_index("method")[[
    "test_seen_query_mean_iou", "test_unseen_query_mean_iou"
]].mul(100).plot.bar(ax=axes[1], color=("#818cf8", "#f59e0b"))
axes[1].set(title="Internal ablation (query-mean metric)", ylabel="Mean query IoU (%)", xlabel="")
axes[1].tick_params(axis="x", rotation=35)
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "benchmark_comparison.png", dpi=180, bbox_inches="tight")
plt.show()


## 7. Interpretation rules

- **Competitive** may be claimed only with the qualifier
  “parent-mask-conditioned / Oracle-Obj-like”.
- The comparison is **approximate**, not a strict leaderboard submission,
  because the model's binary-query training and decoding differ from official
  multiclass benchmark systems.
- Pred-All rows are contextual references and are never treated as equivalent
  to a model receiving ground-truth parent masks.
- If `parent_mask_repair_activations > 0`, disclose it as a training-input
  limitation; do not make a strict scientific-hygiene claim without retraining.
- The internal ablation table is the strictest controlled evidence for the
  contribution of UVD, query gating, rotation consistency, and geometry dropout.


In [10]:
expected = (
    "protocol_audit.json", "parent_mask_repair_cases.csv", "parent_mask_repair_audit.json",
    "internal_ablation.csv", "semantic_confusion_matrix.npy", "per_class_iou.csv",
    "benchmark_summary.json", "literature_sources.csv", "literature_comparison.csv",
    "scientific_interpretation.json",
    "benchmark_comparison.png",
)
artifact_table = pd.DataFrame([
    {"file": name, "exists": (OUTPUT_DIR / name).is_file(),
     "size_kb": round((OUTPUT_DIR / name).stat().st_size / 1024, 1) if (OUTPUT_DIR / name).is_file() else None}
    for name in expected
])
display(artifact_table)
assert artifact_table["exists"].all()


,file,exists,size_kb
0,protocol_audit.json,True,1.3
1,parent_mask_repair_cases.csv,True,4.6
2,parent_mask_repair_audit.json,True,0.1
3,internal_ablation.csv,True,0.8
4,semantic_confusion_matrix.npy,True,105.2
5,per_class_iou.csv,True,9.6
6,benchmark_summary.json,True,0.8
7,literature_sources.csv,True,0.5
8,literature_comparison.csv,True,2.0
9,scientific_interpretation.json,True,0.7
